# Step 5: Model Selection & Training

This notebook documents the full model training pipeline:
- **Naive Baseline** (always predict 0)
- **ML Baseline** (Logistic Regression)
- **XGBoost** (algorithm-level class weighting)
- **LightGBM** (algorithm-level class weighting)

All models are serialized to the `models/` directory after training.

In [ ]:
import pandas as pd
import numpy as np
import sys, os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score, ConfusionMatrixDisplay
import xgboost as xgb
import lightgbm as lgb
import joblib
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('..'))

In [ ]:
# --- Load Processed Data ---
# If the full ~1GB CSV is too slow, change nrows to a smaller sample for testing.
# Remove nrows entirely to run on the full 6.3M row dataset.
df = pd.read_csv('../data/processed/processed_transactions.csv')
print(f'Loaded shape: {df.shape}')

y = df['isFraud']
X = df.drop(columns=['isFraud'])
print(f'Fraud rate: {y.mean()*100:.4f}%')

## 1. Data Splitting (80/20 Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_weight = float(neg / pos)

print(f'Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows')
print(f'scale_pos_weight = {scale_weight:.2f} (Negatives / Positives)')

## 2. Naive Baseline

In [ ]:
y_naive = np.zeros(len(y_test), dtype=int)
print('--- Naive Baseline (Predict All Legitimate) ---')
print(classification_report(y_test, y_naive))

## 3. ML Baseline — Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print('--- Logistic Regression Baseline ---')
print(classification_report(y_test, y_pred_lr))
print(f'PR AUC: {average_precision_score(y_test, y_prob_lr):.4f}')

## 4. XGBoost Weighted Classifier

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_weight,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('--- XGBoost ---')
print(classification_report(y_test, y_pred_xgb))
print(f'PR AUC: {average_precision_score(y_test, y_prob_xgb):.4f}')

## 5. LightGBM Weighted Classifier

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

print('--- LightGBM ---')
print(classification_report(y_test, y_pred_lgb))
print(f'PR AUC: {average_precision_score(y_test, y_prob_lgb):.4f}')

## 6. Confusion Matrices (Side-by-Side Comparison)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model, pred, name in zip(
    axes,
    [lr, xgb_model, lgb_model],
    [y_pred_lr, y_pred_xgb, y_pred_lgb],
    ['Logistic Regression', 'XGBoost', 'LightGBM']
):
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/confusion_matrices.png')
plt.show()
print('Saved to reports/figures/confusion_matrices.png')

## 7. Save All Models

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(lr,        '../models/baseline_logreg.pkl')
joblib.dump(xgb_model, '../models/xgboost_model.pkl')
joblib.dump(lgb_model, '../models/lightgbm_model.pkl')
print('All models saved to models/')